# TechCompenso job scraping

Scrape job postings from techcompenso.com using Selenium.

## 1. Imports

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import sqlite3, json as _json, time

## 2. Launch browser and load jobs page

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)
driver.get("https://techcompenso.com/offerte-di-lavoro")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
time.sleep(3)
print(driver.title)

## 3. Collect job cards

In [ ]:
cards = driver.find_elements(By.CSS_SELECTOR, "a")
jobs = []
for c in cards:
    text = c.text.strip()
    href = c.get_attribute("href")
    if text and href and "/offerte-di-lavoro/" in href and href != "https://techcompenso.com/offerte-di-lavoro":
        jobs.append({"title": text, "url": href})
# de-duplicate
seen=set(); uniq=[]
for j in jobs:
    if j["url"] not in seen:
        seen.add(j["url"]); uniq.append(j)
print(f"Found {len(uniq)} postings")
uniq[:5]

## 4. Persist to SQLite

In [ ]:
db_path = "tech_compenso_jobs.db"
conn = sqlite3.connect(db_path)
df = pd.DataFrame(uniq)
if not df.empty and "url" in df.columns:
    df = df.set_index("url")
    df.to_sql("postings", conn, if_exists="replace", index=True, index_label="url")
    conn.commit()
    print(f"Saved {len(df)} postings")
conn.close()
driver.quit()